# Stage 3 — Label Engineering

Loads the pre-computed acute event labels from `data/raw/acute_events.csv`
(generated in Stage 1 Step 10 by `src/acute_events.py`).

**This notebook does NOT derive labels from engineered features.**
The DGP separates label generation from the feature space:
- Background risk: `condition_cluster + age + comorbidity_count`
- Protective moderator: `overall_utilisation_rate` (up to 70% risk reduction)

Nudge features (zero_allied_flag, sessions_remaining_*, high_gp_low_allied)
correlate with utilisation, giving them genuine indirect causal paths to the label.

## Step 1 — Load acute events

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir     = Path().resolve().parent / 'data' / 'raw'
features_dir = Path().resolve().parent / 'data' / 'features'

# Load pre-computed labels. Stage 1 Step 10 must have run first.
acute = pd.read_csv(data_dir / 'acute_events.csv')

print(f'Loaded: {acute.shape}')
print(f'Columns: {list(acute.columns)}')
print(f'\nPositive class rate: {acute["acute_event"].mean():.1%}')
print(f'Events: {int(acute["acute_event"].sum()):,}  |  Non-events: {int((acute["acute_event"]==0).sum()):,}')
print(f'\nscale_pos_weight: {(acute["acute_event"]==0).sum() / acute["acute_event"].sum():.2f}')

Loaded: (50000, 9)
Columns: ['member_id', 'age', 'comorbidity_count', 'condition_cluster', 'overall_utilisation_rate', 'background_risk', 'protection_factor', 'acute_event_prob', 'acute_event']

Positive class rate: 17.6%
Events: 8,816  |  Non-events: 41,184

scale_pos_weight: 4.67


## Step 2 — Validate causal structure

Three checks that confirm the DGP is working as designed:
1. Higher utilisation -> lower acute event rate (protective effect active)
2. Higher comorbidity -> higher acute event rate (amplifier monotone)
3. Cluster ordering correct (Mixed > MSK|Metabolic > MH > Healthy)

In [2]:
# --- 2a. Protective effect: high utilisation -> lower event rate ---
print('=== Acute rate by utilisation quartile ===')
acute['util_quartile'] = pd.qcut(
    acute['overall_utilisation_rate'], q=4,
    labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)']
)
util_rates = acute.groupby('util_quartile', observed=True)['acute_event'].mean()
print(util_rates.round(3))
print(f'  Risk ratio Q1/Q4: {util_rates.iloc[0]/util_rates.iloc[-1]:.2f}x')

assert util_rates.iloc[0] > util_rates.iloc[-1], \
    'Protective effect absent -- low util should have higher event rate'
print('  Protective effect confirmed')

# --- 2b. Comorbidity amplifier: monotone increasing ---
print('\n=== Acute rate by comorbidity count ===')
comorb_rates = acute.groupby('comorbidity_count')['acute_event'].mean().sort_index()
print(comorb_rates.round(3))

for i in range(len(comorb_rates) - 1):
    assert comorb_rates.iloc[i] <= comorb_rates.iloc[i+1], \
        f'Comorbidity amplifier not monotone at count {i}'
print('  Comorbidity amplifier monotone confirmed')

# --- 2c. Cluster ordering ---
print('\n=== Acute rate by condition cluster ===')
cluster_rates = acute.groupby('condition_cluster')['acute_event'].mean().sort_values(ascending=False)
print(cluster_rates.round(3))

assert cluster_rates['Mixed']  > cluster_rates.get('Healthy', 0), 'Mixed should exceed Healthy'
assert cluster_rates['Mixed']  > cluster_rates.get('MSK', 0),     'Mixed should exceed MSK'
assert cluster_rates.get('MSK', 0) > cluster_rates.get('Healthy', 0), 'MSK should exceed Healthy'
print('  Cluster ordering confirmed')

# --- 2d. Background risk distribution ---
print(f'\n=== Background risk (pre-moderation) ===')
print(acute['background_risk'].describe().round(3))
print(f'\n=== Protection factor distribution ===')
print(acute['protection_factor'].describe().round(3))

=== Acute rate by utilisation quartile ===
util_quartile
Q1 (lowest)     0.227
Q2              0.189
Q3              0.158
Q4 (highest)    0.121
Name: acute_event, dtype: float64
  Risk ratio Q1/Q4: 1.88x
  Protective effect confirmed

=== Acute rate by comorbidity count ===
comorbidity_count
0    0.027
1    0.136
2    0.345
3    0.444
Name: acute_event, dtype: float64
  Comorbidity amplifier monotone confirmed

=== Acute rate by condition cluster ===
condition_cluster
Mixed        0.353
MSK          0.154
Metabolic    0.135
MH           0.114
Healthy      0.027
Name: acute_event, dtype: float64
  Cluster ordering confirmed

=== Background risk (pre-moderation) ===
count    50000.000
mean         0.252
std          0.173
min          0.032
25%          0.160
50%          0.201
75%          0.413
max          0.812
Name: background_risk, dtype: float64

=== Protection factor distribution ===
count    50000.000
mean         0.700
std          0.162
min          0.223
25%          0.583
5

## Step 3 — Save labels.parquet

Rename `acute_event` -> `high_acute_risk` for backward compatibility with
Stages 4 and 5 (model training and scoring). No changes needed downstream.

In [3]:
labels = (
    acute[['member_id', 'acute_event']]
    .rename(columns={'acute_event': 'high_acute_risk'})
)

labels_path = features_dir / 'labels.parquet'
labels.to_parquet(labels_path, index=False)

# Compute and display the scale_pos_weight for Stage 4
pos   = int(labels['high_acute_risk'].sum())
neg   = len(labels) - pos
spw   = neg / pos

print(f'Labels saved: {labels_path}')
print(f'Shape: {labels.shape}')
print(f'Positive rate:    {labels["high_acute_risk"].mean():.1%}  ({pos:,} members)')
print(f'scale_pos_weight: {spw:.2f}  <-- update Stage 4 if this changed')

Labels saved: /home/alex/personal_projects/allied-health-nudge/data/features/labels.parquet
Shape: (50000, 2)
Positive rate:    17.6%  (8,816 members)
scale_pos_weight: 4.67  <-- update Stage 4 if this changed


## Step 4 — Validation checks

In [4]:
loaded = pd.read_parquet(features_dir / 'labels.parquet')

assert len(loaded) == 50_000,                           'Row count wrong'
assert loaded['member_id'].nunique() == 50_000,         'Duplicate member_ids'
assert loaded['high_acute_risk'].isin([0,1]).all(),     'Non-binary label values'
assert 0.15 <= loaded['high_acute_risk'].mean() <= 0.30, \
    f"Positive rate {loaded['high_acute_risk'].mean():.1%} outside expected [15%, 30%]"
assert 'high_acute_risk' in loaded.columns,             'Label column missing'

# Verify Stage 3 file is consistent with the raw acute_events source
raw = pd.read_csv(data_dir / 'acute_events.csv')
assert (raw['acute_event'].values == loaded['high_acute_risk'].values).mean() > 0.99, \
    'Labels.parquet does not match acute_events.csv -- possible join error'

pos_r = loaded['high_acute_risk'].mean()
spw_v = (loaded['high_acute_risk']==0).sum() / loaded['high_acute_risk'].sum()

print('All Stage 3 validation checks passed')
print(f'  Positive rate:    {pos_r:.1%}')
print(f'  scale_pos_weight: {spw_v:.2f}')
print(f'  --> Carry scale_pos_weight = {spw_v:.2f} to Stage 4 LightGBM params')

All Stage 3 validation checks passed
  Positive rate:    17.6%
  scale_pos_weight: 4.67
  --> Carry scale_pos_weight = 4.67 to Stage 4 LightGBM params
